# Tethys-Chloris (T&C): model setup and sensitivity analysis of soil moisture dynamics

## Material
* Case Study Materials
  * Script to create the model inputs

* Input Data
  * Files containing CO2 air concentration

* Visualization Tools
  * Functions for results visualizations

## Objectives

This session is designed to help students learn how to run a basic case study using the plot-scale version of the T&C model (*Fatichi et al., 2012*) with a specific focus of analyzing soil moisture dynamics under variable inputs and conditions. The document outlines the key steps, verifying critical model settings, understanding model inputs and outputs, and conducting model sensitivity analysis of soil moisture dynamics. 

The tutorial here will focus on one case study from Switzerland ([Chamau, ZG](https://www.google.com/maps/place/47%C2%B012'36.8%22N+8%C2%B024'38.3%22E/@47.2102278,8.4099778,273m/data=!3m2!1e3!4b1!4m4!3m3!8m2!3d47.2102269!4d8.4106451?entry=ttu)).

# Part A: input preparation, model setup and validation

## 1. Julia installation

Please make sure you have [installed Julia](https://julialang.org/downloads/) on your PC.

## 2. Understand the meteorological input file

Meteorological input data, such as air temperature, precipitation, and solar radiation, are essential inputs to the ecohydrological model T&C. To prepare the input data required for T&C, we will use data from the [FLUXNET monitoring network](https://fluxnet.org/). Specifically, the Swiss FluxNet, a regional subset of FLUXNET, currently encompasses six long-term ecosystem monitoring sites in Switzerland. The original data has been pre-processed to fit the input format of T&C in Chamau.

* The file **`CH-Cha.nc`** contains the necessary meteorological forcing to run T&C. To understand the meaning of each variable in these files, refer to the **`T&C_Variables_LIST_PlotScale.pdf`** or the slides.

Once you have found the file, consider the following questions:

- *What meteorological data does the file contain?*
- *What is the time period covered by the meteorological data?*
- *What are the mean annual precipitation and mean temperature in Chamau?*
- *To which biome does this site belong according to the Whittaker classification (Whittaker, 1970)?*

### Accessing data stored in the NetCDF file in Julia

To access the file, you will need to load the [`NCDatasets`](https://juliageo.org/NCDatasets.jl/stable/) package in the environment and open the file as follows

```julia
using NCDatasets
netcdf_path = <path-to-the-file>
ds = NCDataset(netcdf_path, "r");
```

The data is stored in the [NetCDF](https://www.unidata.ucar.edu/software/netcdf)(Network Common Data Form) format, which provides convenient informatino about the dimensions of all variables. To access a variable, such as the wind speed, 

```julia
ds["Ws"]
```

## 3. Understand the case study

Open the file **`create_chamau_data.jl`** and take a moment to familiarize yourself with the case study:

- *What is the vegetation type and land cover in Chamau? Is there any land management?*
- *What are the soil depth and soil texture? How many soil layers are set?*

*Hint*: To get a better idea about the vegetation in Chamau, check the variables **ZR95_H**, **ZR95_L**, **aSE_H**, **aSE_L**, **Ccrown**, and **Mpar_L**. The abbreviations and meanings of these variables can be found in **`T&C_Variables_LIST_PlotScale.pdf`**.

Remember to perform the same checks before running the simulation for a new case study. For the case study in Chamau all parameters are already provided.

## 4. Ready to run the simulation

To run the model, initializing the model based on the parameters, forcing inputs and initial conditions

In [ ]:
using TethysChloris

yaml_path = joinpath(@__DIR__, "data", "CH-Cha.yaml");
netcdf_path = joinpath(@__DIR__, "data", "CH-cha.nc");

# Initialize model
FT = Float64
model = initialize_model(FT, netcdf_path, yaml_path);

Then, specify the options for the simulation

In [ ]:
import Roots

options = ModelOptions(
    SoilTemperature = true,
    FreezingSoil = true,
    VegetationSnowInteractions = true,
    OPT_CR = RootsNonBracketingStrategy(FT),
    OPT_ST = RootsBracketingStrategy(FT, Roots.Brent()),
    OPT_ST2 = RootsBracketingStrategy(FT, Roots.Brent()),
    OPT_VD = ODEOptions(; abstol = 0.05),
);

Now that everything is ready, you can run the simulation

In [ ]:
ALB, CK1, Ck, DQ = run_simulation(model; options);

## Result checking

To quickly check some results, you can easily plot variables. For example, try plotting the **O (volumetric soil water content)** using the `plot` function

```julia
using Plots
plot(model.state.hydrologic.O)
```

To get an overview of the results, use the provided plotting functions provided in the [TethysChloris package](https://epfl-enac.github.io/change-tethyschloris-doc/stable/04-plots/). 

# Part B: Sensitivity analysis

In this second part, we will perform a sensitivity analysis of model outputs (particularly soil moisture dynamics – Lecture 2) to variations in soil texture, root depth, and precipitation.

## 1. Sensitivity analysis

Modify the parameters listed in Table 1 and explore how these parameters impact key ecohydrological variables. See Table 2 for an overview of the most important variables. Note that, for some variables, such as runoff, cumulative values might be more informative than means. For illustrative purposes you may consider a sandy soil when varying vegetation properties and precipitation, even though it does not reflect the true condition at the site.


### Effect of soil and vegetation properties, and precipitation on soil moisture dynamics

#### Soil properties

| Variables                           | O [-] | Saturation [-] | ET [mm/h] | Lk [mm/h] | ET [mm] | 
| ----------------------------------- | ----- | -------------- | --------- | --------- | ------- | 
| (Sand) Psan=80%, Pcla=5%            | &nbsp; | &nbsp; | &nbsp; | &nbsp; | &nbsp; | 
| (Silt loam) Psan=25.4%, Pcla=24.4%  | &nbsp; | &nbsp; | &nbsp; | &nbsp; | &nbsp; | 
| (Clay) Psan=25%, Pcla=50%           | &nbsp; | &nbsp; | &nbsp; | &nbsp; | &nbsp; | 

#### Vegetation properties

| Variables                               | O [-] | Saturation [-] | ET [mm/h] | Lk [mm/h] | ET [mm] | 
| --------------------------------------- | ----- | -------------- | --------- | --------- | ------- | 
| Root depth decrease to 100 [mm] (sand)  | &nbsp; | &nbsp; | &nbsp; | &nbsp; | &nbsp; | 
| Root depth increase to 500 [mm] (sand)  | &nbsp; | &nbsp; | &nbsp; | &nbsp; | &nbsp; | 

#### Precipitation

| Variables     | O [-] | Saturation [-] | ET [mm/h] | Lk [mm/h] | ET [mm] | 
| ------------- | ----- | -------------- | --------- | --------- | ------- | 
| Double (sand) | &nbsp; | &nbsp; | &nbsp; | &nbsp; | &nbsp; | 
| Half (sand)   | &nbsp; | &nbsp; | &nbsp; | &nbsp; | &nbsp; |

### Relevant state variables in Tethys-Chloris

| Name | Unit | Description |
| ------------- | ----- | -------------- |
| O | [-] | Soil moisture – Liquid volumetric soil water content |
| Osat | [-] | Water content at saturation | 
| ET | [mm/h] | Total evapotranspiration | 
| Lk | [mm/h] | Bottom leakage soil to bedrock | 
| Rd | [mm/h] | Saturation excess runoff | 
| Rh | [mm/h] | Infiltration excess runoff | 



## 2.  Interpreting and plotting the results
In order to get a better understanding of the soil moisture dynamics, let’s take a closer look at how the variables above evolve over time.
* Plot the soil moisture in time at different depths, as well as ET, Lk, and Rh. How do these variables relate to each other?
* Plot the time-averaged soil moisture at different depths. Can you explain the differences between the scenarios from the sensitivity analysis? For instance, how does soil texture affect moisture, and why? Is there anything unexpected? You can use the code provided in `prepare_plots.m`.
* Can you compute for the different cases how much water you have in the root zone? How does this change with root depth/precipitation/soil type?